# Day 7. Backpropagation / Optimizer / 간단한 Neural Network 학습

**1주차 마지막 날입니다!**

**오늘의 목표**
- Gradient가 "Loss를 줄이려면 Weight를 **어느 방향으로, 얼마나** 움직여야 하는가"라는 걸 이해한다
- Gradient Descent를 Optimizer 없이 **직접** 구현해본다
- `loss.backward()`, `.grad`, `optimizer.zero_grad()`, `optimizer.step()`이 각각 무슨 일을 하는지 안다
- 작은 신경망을 처음부터 끝까지 학습시켜본다
- 1주차 전체 흐름(입력 → ... → Weight Update)을 설명할 수 있다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.tolist()]

## 1. Gradient = 기울기 = "어느 쪽으로 가야 내려가나?"

Day 6에서 Loss는 "모델이 얼마나 틀렸나"를 숫자 하나로 나타낸 것이라고 했습니다.
학습의 목표는 **Loss를 최대한 작게 만드는 Weight를 찾는 것**이에요.

가장 단순한 예로 시작합시다. Weight가 `w` 하나뿐이고, Loss가 이렇게 생겼다고 해요.

```text
loss(w) = (w - 3)²
```

`w = 3`일 때 Loss가 0으로 가장 작습니다. 그런데 **모델은 이 답을 모릅니다.**
모델이 알 수 있는 건 "지금 위치에서 **기울기**가 어떤가"뿐이에요.

```text
기울기(gradient) = d loss / d w = 2 × (w - 3)
```

| 지금 w | 기울기 | 의미 | 그래서 w를 |
|---|---|---|---|
| 0 | 2×(0-3) = **-6** | 오른쪽으로 갈수록 Loss가 **내려감** | **키운다** |
| 5 | 2×(5-3) = **+4** | 오른쪽으로 갈수록 Loss가 **올라감** | **줄인다** |
| 3 | **0** | 평평함 = 바닥 | 그대로 |

> 💡 **핵심**: 기울기의 **반대 방향**으로 가면 Loss가 줄어듭니다. 기울기가 클수록 바닥에서 멀다는 뜻이에요.

In [ ]:
w_axis = torch.linspace(-1, 7, 200)
loss_curve = (w_axis - 3) ** 2

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(w_axis, loss_curve, linewidth=2.5, label=r"loss(w) = $(w - 3)^2$")

for w0, color in [(0.0, "tab:red"), (5.0, "tab:green")]:
    l0 = (w0 - 3) ** 2
    g0 = 2 * (w0 - 3)
    xs = torch.linspace(w0 - 1.5, w0 + 1.5, 20)
    ax.plot(xs, l0 + g0 * (xs - w0), "--", color=color, linewidth=2)        # 접선
    ax.scatter([w0], [l0], color=color, s=80, zorder=5)
    ax.annotate(f"w={w0:.0f}, 기울기={g0:+.0f}", (w0, l0), textcoords="offset points",
                xytext=(10, 10), color=color, fontsize=11)

ax.scatter([3], [0], color="black", s=80, zorder=5)
ax.annotate("바닥 (w=3)", (3, 0), textcoords="offset points", xytext=(-20, 12))
ax.set_xlabel("w")
ax.set_ylabel("loss")
ax.set_ylim(-2, 18)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

점선이 **접선**(그 지점의 기울기)입니다.
- 빨간 점(w=0): 접선이 오른쪽 아래로 내려감 → 오른쪽으로 가야 함
- 초록 점(w=5): 접선이 오른쪽 위로 올라감 → 왼쪽으로 가야 함

## 2. Gradient Descent를 직접 해보기

규칙은 딱 한 줄입니다.

```text
w ← w - learning_rate × gradient
```

- `- gradient`: 기울기 **반대 방향**으로
- `learning_rate` (학습률): 한 번에 **얼마나 크게** 움직일지

Optimizer 없이 반복문으로 직접 해봅시다.

In [ ]:
w = 0.0
lr = 0.1
history = [w]

for step in range(25):
    grad = 2 * (w - 3)        # 기울기 계산 (손으로 구한 공식)
    w = w - lr * grad         # 반대 방향으로 조금 이동
    history.append(w)
    if step < 5 or step == 24:
        print(f"step {step:2d}: grad={grad:+.3f}  →  w={w:.4f}  loss={(w-3)**2:.5f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(w_axis, loss_curve, linewidth=2, color="lightgray")
hs = torch.tensor(history)
axes[0].plot(hs, (hs - 3) ** 2, "o-", color="tab:blue", markersize=5)
axes[0].set_title("Loss 곡선 위에서 w가 내려가는 모습", fontsize=12)
axes[0].set_xlabel("w"); axes[0].set_ylabel("loss"); axes[0].grid(alpha=0.3)

axes[1].plot(history, "o-", markersize=4)
axes[1].axhline(3, color="black", linestyle="--", label="정답 w=3")
axes[1].set_title("step마다 w의 변화", fontsize=12)
axes[1].set_xlabel("step"); axes[1].set_ylabel("w"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

처음엔 기울기가 커서 성큼성큼 움직이고, 바닥에 가까워질수록 기울기가 작아져서 **조금씩** 움직입니다.
모델은 정답(3)을 몰랐지만 기울기만 보고 찾아냈어요.

## 3. Learning Rate: 너무 작아도, 너무 커도 문제

같은 문제를 Learning Rate만 바꿔서 해봅시다.

In [ ]:
def run_gd(lr, steps=25, w=0.0):
    hist = [w]
    for _ in range(steps):
        w = w - lr * 2 * (w - 3)
        hist.append(w)
    return hist

plt.figure(figsize=(8, 4))
for lr_, label in [(0.01, "lr=0.01 (너무 작음)"), (0.1, "lr=0.1 (적당)"),
                   (0.9, "lr=0.9 (큼: 왔다갔다)"), (1.05, "lr=1.05 (너무 큼: 발산)")]:
    plt.plot(run_gd(lr_), "o-", markersize=3, label=label)
plt.axhline(3, color="black", linestyle="--", linewidth=1)
plt.ylim(-6, 12)
plt.xlabel("step"); plt.ylabel("w")
plt.title("Learning Rate에 따른 w의 변화 (정답 w=3)", fontsize=12)
plt.legend(); plt.grid(alpha=0.3)
plt.show()

| Learning Rate | 결과 |
|---|---|
| 너무 작음 | 방향은 맞는데 **너무 느림** |
| 적당함 | 빠르고 안정적으로 도착 |
| 큼 | 바닥을 **지나쳐서** 왔다갔다 |
| 너무 큼 | 지나칠 때마다 더 멀어져서 **발산** (Loss가 폭발, `nan`) |

> 💡 LLM 학습에서 Loss가 갑자기 `nan`이 되면 가장 먼저 의심하는 게 Learning Rate예요.

## 4. PyTorch가 Gradient를 자동으로 계산해준다: `backward()`

위에서는 `grad = 2 * (w - 3)` 공식을 **손으로** 구했습니다.
하지만 GPT에는 Weight가 수억 개 있어요. 손으로 미분할 수 없습니다.

PyTorch는 **계산 과정을 기록해두었다가** `backward()` 한 줄로 모든 Gradient를 자동으로 계산합니다.

In [ ]:
w = torch.tensor(0.0, requires_grad=True)   # "이 값의 Gradient를 추적해줘"

loss = (w - 3) ** 2
loss.backward()                              # Gradient 계산!

print("loss   :", loss.item())
print("w.grad :", w.grad.item(), "  ← 손으로 구한 2×(0-3) = -6 과 같음")

**세 가지 키워드**
- `requires_grad=True`: "이 텐서는 학습할 값이니까 Gradient를 구해줘" (Day 3에서 본 `nn.Linear`의 weight에는 이미 켜져 있었죠)
- `loss.backward()`: Loss에서 출발해 **거꾸로(backward)** 거슬러 올라가며 Gradient를 계산
- `w.grad`: 계산된 Gradient가 저장되는 곳

## 5. Backpropagation: 왜 "거꾸로"인가? (Chain Rule 감만 잡기)

신경망은 계산이 **여러 단계로 이어져** 있습니다. 2단계짜리 예시를 봅시다.

```text
x = 2 (입력, 고정)
h = w × x        (1단계: Linear 같은 것)
loss = (h - 10)² (2단계: Loss)
```

`w`를 조금 바꾸면 Loss가 얼마나 바뀔까요? **두 단계의 영향을 곱하면** 됩니다.

```text
w가 1 늘면 → h는 x(=2)만큼 늘고            : dh/dw = x = 2
h가 1 늘면 → loss는 2×(h-10)만큼 늘어남    : dloss/dh = 2×(h-10)

dloss/dw = dloss/dh × dh/dw      ← 이게 Chain Rule (연쇄 법칙)
```

**Loss 쪽(뒤)에서부터 한 단계씩 곱해 나가기 때문에** Back-propagation(역전파)이라고 부릅니다.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
x = 2.0

h = w * x
loss = (h - 10) ** 2
loss.backward()

dloss_dh = 2 * (h.item() - 10)
dh_dw = x
print(f"h = {h.item()},  loss = {loss.item()}")
print(f"손으로: dloss/dh × dh/dw = {dloss_dh} × {dh_dw} = {dloss_dh * dh_dw}")
print(f"autograd: w.grad = {w.grad.item()}")

GPT는 이 단계가 수백 개 이어져 있을 뿐, 원리는 똑같습니다. 미분 공식을 외울 필요는 없고,
> **Loss를 줄이는 방향으로 Weight를 수정하기 위해 Gradient를 계산한다**

이것만 이해하면 됩니다. (커리큘럼 1-6의 목표 그대로예요)

## 6. 함정: Gradient는 **쌓인다** → `zero_grad()`가 필요한 이유

`backward()`는 `.grad`에 값을 **덮어쓰지 않고 더합니다.**

In [ ]:
w = torch.tensor(0.0, requires_grad=True)

for i in range(3):
    loss = (w - 3) ** 2
    loss.backward()
    print(f"{i+1}번째 backward 후 w.grad = {w.grad.item()}")

진짜 Gradient는 -6인데 -12, -18로 계속 쌓였죠? 이대로 Weight를 업데이트하면 엉뚱하게 크게 움직입니다.
그래서 매 step마다 **Gradient를 0으로 초기화**해야 합니다.

In [ ]:
w = torch.tensor(0.0, requires_grad=True)

for i in range(3):
    w.grad = None              # 초기화 (optimizer.zero_grad()가 하는 일)
    loss = (w - 3) ** 2
    loss.backward()
    print(f"{i+1}번째 backward 후 w.grad = {w.grad.item()}")

## 7. Optimizer: Weight 업데이트를 대신 해주는 도구

`w = w - lr * w.grad`를 Weight 수억 개에 대해 직접 쓰기는 힘들죠. Optimizer가 해줍니다.

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.SGD([w], lr=0.1)

for step in range(25):
    loss = (w - 3) ** 2
    optimizer.zero_grad()   # 1) Gradient 초기화
    loss.backward()         # 2) Gradient 계산
    optimizer.step()        # 3) w ← w - lr × grad

print("SGD로 찾은 w:", round(w.item(), 4))
print("2절에서 손으로 찾은 w:", round(history[-1], 4), " ← 똑같음")

### 7-1. Optimizer 종류

| Optimizer | 아이디어 | 어디에 |
|---|---|---|
| **SGD** | `w ← w - lr × grad` 그대로 | 2절에서 직접 한 것 |
| **Adam** | 지금까지 움직인 방향(관성)과 Gradient 크기를 기억해서 Weight마다 **보폭을 자동 조절** | 대부분의 딥러닝 기본값 |
| **AdamW** | Adam + Weight가 너무 커지지 않게 살짝 줄여줌(weight decay) | **GPT 등 거의 모든 LLM** |

지금은 "AdamW가 LLM의 표준이다" 정도만 기억하면 됩니다. 4주차 mini GPT도 AdamW로 학습할 거예요.

In [ ]:
def fit(opt_cls, lr, steps=60):
    w = torch.tensor(0.0, requires_grad=True)
    opt = opt_cls([w], lr=lr)
    hist = []
    for _ in range(steps):
        loss = (w - 3) ** 2
        opt.zero_grad(); loss.backward(); opt.step()
        hist.append(w.item())
    return hist

plt.figure(figsize=(8, 3.5))
plt.plot(fit(torch.optim.SGD, 0.05), label="SGD (lr=0.05)", linewidth=2)
plt.plot(fit(torch.optim.Adam, 0.3), label="Adam (lr=0.3)", linewidth=2)
plt.plot(fit(torch.optim.AdamW, 0.3), "--", label="AdamW (lr=0.3)", linewidth=2)
plt.axhline(3, color="black", linestyle=":", linewidth=1)
plt.xlabel("step"); plt.ylabel("w"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Optimizer별로 w=3을 찾아가는 모습", fontsize=12)
plt.show()

이렇게 단순한 문제에서는 Adam이 오히려 바닥을 지나쳤다 돌아오기도 합니다. Adam은 관성이 있어서 그래요.
Adam의 진짜 장점은 **Weight가 수백만 개이고, Weight마다 Gradient 크기가 제각각일 때** 나타납니다.
Weight마다 보폭을 따로 조절해주니까 Learning Rate 하나로도 전체가 고르게 학습돼요.

## 8. 표준 학습 루프 (커리큘럼 1-7)

이제 커리큘럼에 나온 학습 루프를 **한 줄씩** 이해할 수 있습니다.

```python
for step in range(1000):
    prediction = model(x)                    # ① Forward: 입력 → 예측
    loss = loss_function(prediction, y)      # ② Loss: 예측이 정답과 얼마나 다른가
    optimizer.zero_grad()                    # ③ 지난 step의 Gradient 지우기
    loss.backward()                          # ④ Backward: 모든 Weight의 Gradient 계산
    optimizer.step()                         # ⑤ Weight Update: Gradient 반대 방향으로 이동
```

이 5줄은 mini GPT를 포함해 **거의 모든 딥러닝 학습 코드에 똑같이** 들어갑니다.

> 💡 Day 3의 섭씨→화씨, Day 4의 sin 곡선 실험에서 "지금은 결과만 보세요"라고 했던 코드가 바로 이 루프였어요.
> 다시 올라가서 보면 이제 한 줄 한 줄 읽힐 거예요.

## 9. 종합 실습: 작은 신경망으로 분류하기

1주차에 배운 걸 전부 써서 신경망을 처음부터 끝까지 학습시켜 봅시다.

**문제**: 2차원 평면 위의 점들이 두 개의 소용돌이(spiral) 모양으로 섞여 있습니다. 점의 좌표 `(x, y)`만 보고 **빨강(0)인지 파랑(1)인지** 맞히기.

### 9-1. 데이터 만들기

In [ ]:
def make_spirals(n_per_class=200, noise=0.15):
    t = torch.linspace(0.3, 3 * math.pi, n_per_class)
    xs, ys = [], []
    for label in (0, 1):
        angle = t + label * math.pi             # 두 번째 소용돌이는 반 바퀴 돌림
        pts = torch.stack([t * torch.cos(angle), t * torch.sin(angle)], dim=1) / 3
        pts = pts + noise * torch.randn_like(pts)
        xs.append(pts)
        ys.append(torch.full((n_per_class,), label))
    return torch.cat(xs), torch.cat(ys)

X, Y = make_spirals()
print("X:", tuple(X.shape), " ← 점 400개, 좌표 2개")
print("Y:", tuple(Y.shape), " ← 정답 (0 또는 1)")

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], c=Y, cmap="coolwarm", s=12)
plt.title("두 소용돌이 데이터", fontsize=12)
plt.axis("equal")
plt.show()

직선 하나로는 절대 나눌 수 없는 모양이죠? Day 4에서 배운 대로 **Activation이 꼭 필요**한 문제입니다.

### 9-2. 모델: Linear → GELU → Linear → GELU → Linear

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(2, 64),     # 좌표 2개 → 64차원
    nn.GELU(),
    nn.Linear(64, 64),
    nn.GELU(),
    nn.Linear(64, 2),     # 64차원 → 클래스 2개의 점수(logits)
)
print(model)
print("파라미터 개수:", sum(p.numel() for p in model.parameters()))

logits = model(X)
print()
print("logits shape:", tuple(logits.shape), " ← 점마다 [빨강 점수, 파랑 점수]")
print("학습 전 정확도:", round((logits.argmax(dim=-1) == Y).float().mean().item() * 100, 1), "%")

### 9-3. 학습 루프 실행

Loss는 Day 6에서 배운 **Cross Entropy**, Optimizer는 **AdamW**입니다.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
losses = []

for step in range(1500):
    logits = model(X)                       # ① Forward
    loss = F.cross_entropy(logits, Y)       # ② Loss
    optimizer.zero_grad()                   # ③ Gradient 초기화
    loss.backward()                         # ④ Backward
    optimizer.step()                        # ⑤ Weight Update
    losses.append(loss.item())

    if step % 300 == 0 or step == 1499:
        acc = (logits.argmax(dim=-1) == Y).float().mean().item()
        print(f"step {step:4d}  loss {loss.item():.4f}  정확도 {acc*100:5.1f}%")

### 9-4. 결과 보기

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(losses)
axes[0].set_title("Loss 곡선 (step이 지날수록 줄어듦)", fontsize=12)
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss")
axes[0].grid(alpha=0.3)

# 평면 위 모든 점에 대해 모델이 어느 쪽이라고 하는지 칠하기
gx, gy = torch.meshgrid(torch.linspace(-3.5, 3.5, 200), torch.linspace(-3.5, 3.5, 200), indexing="xy")
grid = torch.stack([gx.flatten(), gy.flatten()], dim=1)
with torch.no_grad():
    p_blue = F.softmax(model(grid), dim=-1)[:, 1].reshape(200, 200)

axes[1].contourf(gx, gy, p_blue, levels=20, cmap="coolwarm", alpha=0.5)
axes[1].scatter(X[:, 0], X[:, 1], c=Y, cmap="coolwarm", s=10, edgecolors="k", linewidths=0.3)
axes[1].set_title("모델이 배운 경계 (배경색 = 파랑일 확률)", fontsize=12)
axes[1].set_aspect("equal")
plt.tight_layout()
plt.show()

모델은 "소용돌이"라는 규칙을 들은 적이 없습니다. **랜덤 Weight에서 시작해서**
Forward → Loss → Backward → Update를 1500번 반복한 것만으로 소용돌이 모양의 경계를 스스로 찾아냈어요.

## 10. LLM과의 연결

mini GPT 학습도 **완전히 같은 루프**입니다. 바뀌는 건 모델과 데이터뿐이에요.

| | 오늘 실습 | GPT |
|---|---|---|
| 입력 | 점의 좌표 `(400, 2)` | Token들 `(B, T)` |
| 모델 | Linear + GELU 3층 | Embedding + Transformer Block × N + Linear |
| 출력 logits | `(400, 2)`: 클래스 2개 점수 | `(B, T, V)`: Vocabulary 전체 점수 |
| Loss | Cross Entropy | Cross Entropy (다음 Token 맞히기) |
| Optimizer | AdamW | AdamW |
| 학습 루프 | 5줄 | **같은 5줄** (커리큘럼 6-10) |

```python
# 커리큘럼 6-10: mini GPT 학습
for step in range(max_steps):
    logits, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

## 11. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** `loss(w) = (w - 5)²`일 때, `w = 2`에서의 Gradient를 **손으로** 계산하세요.
(1절의 공식 `2 × (w - 정답)`을 떠올려보세요) 그다음 autograd로 확인합니다.

In [ ]:
my_grad = "?"   # TODO: 숫자 하나

w = torch.tensor(2.0, requires_grad=True)
loss = (w - 5) ** 2
loss.backward()

assert my_grad == w.grad.item(), "부호와 크기를 다시 확인해보세요"
print("Q1 정답!")

**Q2.** Optimizer 없이 Gradient Descent를 직접 구현하세요. `loss(w) = (w + 2)²`의 최솟값(w = -2)을 찾아야 합니다.
- `w.grad`를 이용해서 업데이트
- 업데이트는 `with torch.no_grad():` 안에서 해야 합니다 (업데이트 계산 자체는 기록하지 않도록)
- 매 step Gradient 초기화도 잊지 마세요

In [ ]:
w = torch.tensor(5.0, requires_grad=True)
lr = 0.1

for step in range(100):
    loss = (w + 2) ** 2
    loss.backward()
    with torch.no_grad():
        pass        # TODO: w를 업데이트 (힌트: w -= ...)
    pass            # TODO: Gradient 초기화 (힌트: 6절)

assert abs(w.item() - (-2)) < 1e-3, "w가 -2에 도착하지 못했어요"
print("Q2 정답! w =", round(w.item(), 4))

**Q3.** 아래 학습 루프에서 빠진 줄들을 채워서 Day 3의 섭씨→화씨 변환(`w=1.8, b=32`)을 학습시키세요.

In [ ]:
torch.manual_seed(0)
C = torch.tensor([[-40.0], [0.0], [10.0], [37.0], [100.0]])
F_true = 1.8 * C + 32

model_cf = nn.Linear(1, 1)
optimizer = torch.optim.Adam(model_cf.parameters(), lr=0.5)

for step in range(3000):
    prediction = None                               # TODO: ① Forward
    loss = ((prediction - F_true) ** 2).mean()      # ② Loss
    # TODO: ③ ④ ⑤ 세 줄

w_, b_ = model_cf.weight.item(), model_cf.bias.item()
assert abs(w_ - 1.8) < 0.01 and abs(b_ - 32) < 0.1, "w=1.8, b=32에 도착하지 못했어요"
print(f"Q3 정답! w={w_:.3f}, b={b_:.3f}")

**Q4.** 아래 코드는 `zero_grad()`를 빼먹었습니다. 실행하기 **전에** 결과를 예측해서 고르세요.

- `"A"`: 정상 학습과 똑같이 w=3 근처에 도착한다
- `"B"`: Gradient가 쌓여서 w가 3 근처에 머물지 못하고 크게 출렁인다
- `"C"`: 에러가 나서 실행이 안 된다

In [ ]:
my_answer = "?"   # TODO: "A", "B", "C" 중 하나

w = torch.tensor(0.0, requires_grad=True)
opt = torch.optim.SGD([w], lr=0.1)
ws = []
for step in range(30):
    loss = (w - 3) ** 2
    loss.backward()          # zero_grad() 없음!
    opt.step()
    ws.append(w.item())

actual = "B" if max(abs(v - 3) for v in ws[10:]) > 0.5 else "A"
assert my_answer == actual, "6절을 다시 읽어보세요"
print("Q4 정답! 마지막 w 값들:", [round(v, 2) for v in ws[-5:]])

## 12. 1주차 정리: 1단계 완료 기준

커리큘럼의 1단계 완료 기준은 아래 흐름을 **말로** 설명할 수 있는 것입니다.

```text
입력                 (Day 1: Tensor, Shape)
 ↓
Linear               (Day 2, 3: 행렬 곱, y = xW + b)
 ↓
Activation           (Day 4: GELU)
 ↓
Linear
 ↓
Softmax              (Day 5: 점수 → 확률)
 ↓
Prediction
 ↓
Loss                 (Day 6: Cross Entropy)
 ↓
Backward             (Day 7: Gradient 계산)
 ↓
Weight Update        (Day 7: Optimizer)
```

오늘 9절의 소용돌이 실습이 정확히 이 흐름이었어요. (`F.cross_entropy`가 Softmax까지 안에서 해줍니다)

## 13. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. Gradient는 무엇을 알려주는가? 왜 Gradient의 **반대 방향**으로 Weight를 움직이는가?
2. Learning Rate가 너무 크면/너무 작으면 어떻게 되는가?
3. Backpropagation은 왜 필요한가? 왜 "거꾸로" 계산하는가?
4. `optimizer.zero_grad()`를 빼먹으면 어떻게 되는가?
5. Optimizer는 무엇을 하는가? GPT는 주로 어떤 Optimizer를 쓰는가?
6. **(1주차 총정리)** 위 12절의 흐름을 처음부터 끝까지 자기 말로 설명해보세요.

**내 답:**

1.
2.
3.
4.
5.
6.